# 01b. ML-1M
Тот же протокол, что для Amazon (см. `build_splits`). В ML-1M много оценок с одинаковым `timestamp`:
порядок внутри такой группы определяется порядком строк в файле, из-за чего leave-one-out слегка шумит.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import importlib
import io
import os
import sys
import urllib.request
import zipfile

import pandas as pd

try:
    default_work_dir = '/content/drive/MyDrive/code2repo/kan_rec'
except ImportError:
    default_work_dir = os.path.abspath('./kan_rec')

WORK_DIR = os.environ.get('KAN_REC_DIR', default_work_dir)
DATASET = 'ml1m'

sys.path.insert(0, WORK_DIR)
import common
importlib.reload(common)

paths = common.dataset_paths(WORK_DIR, DATASET)
os.makedirs(paths['results'], exist_ok=True)
device = common.check_gpu()

CUDA недоступна, расчёт пойдёт на CPU


In [3]:
ML1M_URL = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
zip_path = os.path.join(WORK_DIR, 'ml-1m.zip')

if not os.path.exists(zip_path):
    urllib.request.urlretrieve(ML1M_URL, zip_path)

with zipfile.ZipFile(zip_path) as archive:
    ratings_file = io.BytesIO(archive.read('ml-1m/ratings.dat'))
df = pd.read_csv(ratings_file, sep='::', engine='python', names=['user', 'item', 'rating', 'ts'])

same_ts_share = df.duplicated(['user', 'ts'], keep=False).mean()
print(f'{len(df):,} строк, users={df.user.nunique():,}, items={df.item.nunique():,}')
print(f"дубликаты (user, item): {df.duplicated(['user', 'item']).sum()}, "
      f'доля строк с совпадающим ts у пользователя: {same_ts_share:.3f}')

1,000,209 строк, users=6,040, items=3,706
дубликаты (user, item): 0, доля строк с совпадающим ts у пользователя: 0.774


In [4]:
data = common.build_splits(df, k_core=5, rating_min=0, max_users=None, seed=42, name=DATASET)
common.save_data(data, paths['data'])
common.describe_data(data)
print('сохранено:', paths['data'])

users=6,040  items=3,416  interactions=999,611
density=4.8448%  avg len=165.5  p50/p90/p99=[ 96.   400.   906.05]  min len=18
сохранено: /content/drive/MyDrive/code2repo/kan_rec/data_ml1m_r0_k5_uNone.pkl


In [5]:
common.run_baselines(data, paths['results'], device)

Random val {'Recall@10': 0.0025, 'NDCG@10': 0.0014, 'Recall@20': 0.0056, 'NDCG@20': 0.0022}
Random test {'Recall@10': 0.0031, 'NDCG@10': 0.0014, 'Recall@20': 0.0075, 'NDCG@20': 0.0024}
Popularity val {'Recall@10': 0.0429, 'NDCG@10': 0.0207, 'Recall@20': 0.0712, 'NDCG@20': 0.0278}
Popularity test {'Recall@10': 0.0369, 'NDCG@10': 0.018, 'Recall@20': 0.0682, 'NDCG@20': 0.0258}
ориентир для Random: Recall@10 ≈ 0.002927
